<img src="http://imgur.com/1ZcRyrc.png" style="float: left; margin: 15px; height: 80px">

# Austen vs Poe Model Creation
This notebook will serve to create a model that determines whether an input is more similar to Edgar Allan Poe or Jane Austen. We then save this model using **pickling** so that it can be imported and deployed on a `streamlit` app.

In [1]:
import os
import numpy as np
import pandas as pd
import pickle

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline

## Data Import
Some collected works of both Jane Austen and Edgar Allen Poe.

In [2]:
df = pd.read_csv('data/austen_poe.csv').dropna()
df.iloc[20:25]

,text,author
22,A continuance in a place where everything remi...,Jane Austen
23,Mrs. John Dashwood did not at all approve of w...,Jane Austen
24,"""It was my father's last request to me,"" repli...",Jane Austen
25,"""He did not know what he was talking of, I dar...",Jane Austen
26,"""He did not stipulate for any particular sum, ...",Jane Austen


In [3]:
df.tail(5)

,text,author
19035,But to these words I hearkened in vain for a r...,Edgar Allan Poe
19036,"""Fortunato!""",Edgar Allan Poe
19037,No answer. I called again--,Edgar Allan Poe
19038,"""Fortunato--""",Edgar Allan Poe
19039,No answer still. I thrust a torch through the...,Edgar Allan Poe


In [4]:
df.author.value_counts(normalize=True)

Jane Austen        0.607353
Edgar Allan Poe    0.392647
Name: author, dtype: float64

## Preprocessing and Modeling

In [5]:
X = df['text']
y = df['author']

tts = train_test_split(X, y, random_state=111)

In [6]:
count_vec = ('cv', CountVectorizer())
tfidf_vec = ('tf', TfidfVectorizer())
logreg = ('lr', LogisticRegression(max_iter=10000, penalty='none'))
nb = ('nb', MultinomialNB())

In [7]:
pipes = [
    Pipeline([count_vec, logreg]),
    Pipeline([tfidf_vec, logreg]),
    Pipeline([count_vec, nb]),
    Pipeline([tfidf_vec, nb])
]

In [8]:
def evaluate_pipe(pipe, tts, lbl):
    X_train, X_test, y_train, y_test = tts
    pipe.fit(X_train, y_train)
    train_score = pipe.score(X_train, y_train)
    test_score = pipe.score(X_test, y_test)
    print('='*40)
    print(f"Pipeline #{i}:")    
    print(f"Train Acc = {100*train_score:,.2f}% :: Test Acc = {100*test_score:,.2f}%")

In [9]:
for i, pipe in enumerate(pipes):
    evaluate_pipe(pipe, tts, i)

Pipeline #0:
Train Acc = 99.64% :: Test Acc = 94.41%
Pipeline #1:
Train Acc = 99.64% :: Test Acc = 95.25%
Pipeline #2:
Train Acc = 95.11% :: Test Acc = 93.97%
Pipeline #3:
Train Acc = 92.05% :: Test Acc = 88.91%


In [10]:
if not os.path.exists('models'):
    os.mkdir('models')
    
with open('models/austen-poe-detector.pkl', 'wb') as f:
    pickle.dump(pipe[1], f)    